# Logistic Regression — Structural Damage Classification

Uses the existing **scaled HOG features** from `results/`. No HOG extraction, SVD, or scaling is repeated.


## 1. Setup


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

SEED = 42
VAL_FRACTION = 0.15

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "results").exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project root:", PROJECT_ROOT)
print("Imports and configuration loaded.")


Project root: c:\Users\Noothan\Documents\my_work\structural-damage-classification
Imports and configuration loaded.


## 2. Load existing HOG features


In [2]:
X_train_hog = np.load(RESULTS_DIR / "X_train_hog.npy", mmap_mode="r")
X_val_hog = np.load(RESULTS_DIR / "X_val_hog.npy", mmap_mode="r")
X_test_hog = np.load(RESULTS_DIR / "X_test_hog.npy", mmap_mode="r")

print("HOG features loaded.")
print("X_train_hog:", X_train_hog.shape)
print("X_val_hog  :", X_val_hog.shape)
print("X_test_hog :", X_test_hog.shape)


HOG features loaded.
X_train_hog: (10039, 26244)
X_val_hog  : (1772, 26244)
X_test_hog : (1460, 26244)


## 3. Load labels and recreate the exact split


In [3]:
Y_TRAIN_PATH = DATA_DIR / "task2_damage_state_1" / "task2_y_train.npy"
Y_TEST_PATH = DATA_DIR / "task2_damage_state_1" / "task2_y_test.npy"

y_train_raw = np.load(Y_TRAIN_PATH)
y_test_raw = np.load(Y_TEST_PATH)

y_train_all = np.argmax(y_train_raw, axis=1)
y_test = np.argmax(y_test_raw, axis=1)

all_indices = np.arange(len(y_train_all))
train_idx, val_idx = train_test_split(
    all_indices,
    test_size=VAL_FRACTION,
    random_state=SEED,
    stratify=y_train_all
)

y_train = y_train_all[train_idx]
y_val = y_train_all[val_idx]

print("Train:", len(y_train))
print("Validation:", len(y_val))
print("Test:", len(y_test))


Train: 10039
Validation: 1772
Test: 1460


## 4. Verify


In [4]:
assert X_train_hog.shape[0] == len(y_train)
assert X_val_hog.shape[0] == len(y_val)
assert X_test_hog.shape[0] == len(y_test)

print("✓ Features and labels match.")
print("✓ Ready for Logistic Regression.")


✓ Features and labels match.
✓ Ready for Logistic Regression.


## 5. Train Logistic Regression

This is the baseline model. It uses the existing scaled HOG features directly.


In [ ]:
print("Training Logistic Regression baseline...")

logreg_baseline = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="saga",
    max_iter=150,
    tol=1e-2,
    random_state=SEED,
    verbose=1
)

logreg_baseline.fit(X_train_hog, y_train)

print("✓ Logistic Regression training completed.")


Training Logistic Regression baseline...


c:\Users\Noothan\Documents\my_work\structural-damage-classification\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 6. Validation evaluation


In [ ]:
y_val_pred = logreg_baseline.predict(X_val_hog)

print(f"Validation Accuracy : {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Validation Precision: {precision_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"Validation Recall   : {recall_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"Validation F1       : {f1_score(y_val, y_val_pred, zero_division=0):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred,
    target_names=["Class 0", "Class 1"],
    zero_division=0
))


## 7. Final test evaluation


In [ ]:
y_test_pred = logreg_baseline.predict(X_test_hog)

print(f"Test Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test Recall   : {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test F1       : {f1_score(y_test, y_test_pred, zero_division=0):.4f}")

print("\nTest Classification Report:")
print(classification_report(
    y_test, y_test_pred,
    target_names=["Class 0", "Class 1"],
    zero_division=0
))


## 8. Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Class 0", "Class 1"]
)
disp.plot()
plt.title("Logistic Regression — Test Confusion Matrix")
plt.show()


## 9. ROC curve and AUC


In [ ]:
y_test_prob = logreg_baseline.predict_proba(X_test_hog)[:, 1]
test_auc = roc_auc_score(y_test, y_test_prob)
fpr, tpr, _ = roc_curve(y_test, y_test_prob)

print(f"Test ROC-AUC: {test_auc:.4f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {test_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression — ROC Curve")
plt.legend()
plt.grid(True)
plt.show()


## 10. Save model


In [ ]:
import joblib

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "logistic_regression_hog.pkl"
joblib.dump(logreg_baseline, model_path)

print("✓ Model saved to:", model_path)


## Final note

The model uses the same HOG feature pipeline and train/validation/test split as preprocessing. Labels are shown as **Class 0** and **Class 1** until their semantic mapping is verified from the dataset documentation.
